# 05 — Interim JEV screening evaluation summary

**Purpose:** provide colleagues with a reproducible account of the work completed so far and the decisions that need review before validation on **2,000 unseen studies**. This is an exploratory development-cohort report, not a validation report or a final manuscript.

Run this notebook independently from a fresh project `.venv` kernel. Restart the kernel after package changes, then use **Run All in notebook 05**. The report reads saved artifacts directly; it does not execute earlier notebooks. No API credentials, paid calls, model downloads or training are involved. No files are written unless the optional final export is explicitly enabled in both YAML and code.

| Item | Source |
| --- | --- |
| Report settings and illustrative cutoffs | `configs/evaluations/interim_summary.yaml` |
| Archived run selection | `configs/evaluations/analysis_full.yaml` and its two full workflow profiles |
| Evidence | Prepared states, separate reference CSV, conversion/cohort manifests and saved predictions |
| Outputs | Aggregate tables, figures, a methods draft and an optional private report bundle |
| Future validation | Planned after colleague review; no unseen-study data are loaded here |

Counts, metrics and run metadata below are calculated from the saved files. Citation text, study IDs, ITEM_ID values, raw API responses and credentials are not displayed. Source notebooks must be committed with outputs cleared.

## 1. Load the interim report and check prerequisites

This cell checks that the archived scientific inputs and model identities match the frozen cohort. Both models must use the same eligibility criteria and cohort. Missing or incompatible files cause an informative prerequisite error; the report does not substitute demo data.

**Input:** the report YAML below. **Output:** `summary`, an in-memory object containing aggregate tables, methods text, provenance and the validated development results. The future validation settings are metadata only.

In [ ]:
try:
    from jev_eval.environment import find_project_root
    from jev_eval.summary import load_interim_summary
except ModuleNotFoundError:
    raise RuntimeError(
        "Select this repository's .venv Python kernel with the existing notebook dependencies."
    ) from None

import pandas as pd
from IPython.display import Markdown, display

SUMMARY_CONFIG = "configs/evaluations/interim_summary.yaml"
summary = load_interim_summary(find_project_root(), SUMMARY_CONFIG)
print("Report generated (UTC):", summary.provenance["generated_at"])
print("Status: INTERIM / EXPLORATORY — colleague review pending")
display(summary.tables["review_status"])

## 2. Workflow completed so far and study flow

The project separates preparation, model execution and analysis:

- **00_setup:** environment, dependencies and reproducibility checks.
- **01_prepare_data:** reusable data preparation and validation; the archived Excel conversion produces JSONL states and a separate human-reference CSV.
- **02_model_connection:** single synthetic-state API connection checks.
- **03_run_models:** paired ten-state pilots and subsequent full-cohort execution, with separate saved model runs.
- **04_evaluation:** saved-response analysis, exploratory threshold selection and screening metrics.
- **05_study_summary:** this aggregate interim report for colleague review.

The tables below describe the archived full-cohort evidence. `probabilities_available` counts every usable score; `scored_labelled_pairs` also requires a human label. Unscored studies remain part of the cohort and enter a separate manual-review queue. Counts are per model; do not add them together to obtain the number of unique studies.

In [ ]:
display(summary.tables["study_flow"])
display(summary.tables["reference_summary"].T.rename(columns={0: "value"}))
display(summary.tables["inference_coverage"])

## 3. Data preparation, missingness and discrepancy findings

The workbook was read without changing it. Each JSONL state contains `study_id`, `title` and `abstract`. Human `Inclusion Status` and `ITEM_ID` are stored separately and joined by the original study ID. They are not model-input fields. A missing abstract can still leave a title-only input; only records with no title or abstract are unscored.

The preparation and audit tables come from the checksum-verified conversion manifest, not a new workbook conversion. Missing ITEM_ID values do not prevent a study-ID join. The workbook discrepancy worksheet was reported without automatically changing decisions. Its **label disagreements require colleague review** before describing the human reference standard as adjudicated.

Counts can overlap: missing-text and missing-ITEM_ID warnings are not mutually exclusive. Audit entries can also refer to the same study more than once. If the conversion manifest is unavailable, the report says so rather than inventing preparation findings.

In [ ]:
display(summary.tables["preparation_checks"])
display(summary.tables["audit_summary"].T.rename(columns={0: "value"}))
display(summary.tables["audit_issues"])

## 4. Background: the two models and the binary/hybrid NOUL designs

There are two separate comparison dimensions: **which model answers** and **how the screening questions are organised**. JEV-9B and JEV-27B v1.1 are model deployments. Binary and hybrid NOUL are screening-question designs in this repository. The archived results compare **both models using the same binary design**; they do not compare binary against hybrid screening.

### 4.1 The two hosted JEV models

AutoTrust's JEV models support structured decisions: questions define the possible answers and the service returns probabilities. In this project a **NOUL** is a natural-language yes/no question; its response is the model's probability that the answer is true. The service also supports choice and ordered-score questions, which the current screening runs do not use. See the [JEV-9B API description](https://inference.sh/apps/autotrust/jev-9b) and [JEV-27B v1.1 API description](https://inference.sh/apps/autotrust/jev-27b-v1-1).

| Model | Configured hosted identifier | Background and use here |
| --- | --- | --- |
| JEV-9B | `autotrust/jev-9b` | The smaller JEV model-family deployment. The provider describes text and image support; this evaluation supplies text only. |
| JEV-27B v1.1 | `autotrust/jev-27b-v1-1` | The larger JEV model-family deployment, specifically v1.1. The provider describes this endpoint as a text-only decision model. |

Both receive the same study state and archived eligibility instructions. The output used here is `P(retain_for_fulltext = true)`, followed by an explicitly chosen retention cutoff. These are model probabilities, not a demonstration of calibration on depression-screening studies. A larger model name does not establish better screening performance. Hosted app names also do not pin immutable weights; the original run metadata remains the evidence of what was requested and returned.

Provider descriptions checked **23 November 2026**. Public documentation can change independently of the archived runs.

### 4.2 Binary NOUL versus the hybrid NOUL draft

Here “hybrid” describes a **combination of criterion-level and overall eligibility questions**. Each individual question still has a yes/no probability; it is not a different model architecture or a mixture of `choice`, `score` and NOUL outputs.

| Aspect | Binary NOUL | Hybrid NOUL draft |
| --- | --- | --- |
| Question design | One integrated retention question. | Six criterion-level questions plus a separate integrated retention question. |
| Main question ID | `retain_for_fulltext` | `overall_eligibility` |
| Output | One overall retention probability per study/model. | Seven question-specific probabilities, if a future reviewed workflow executes the draft. |
| Purpose | Directly assess whether the citation should advance to full-text review. | Also describe which eligibility domains the model considers potentially satisfied, to support error review. These judgments are not verified explanations. |
| Evidence in this report | Used for both archived model runs. | Draft only; no hybrid performance results are included. |

The corrected hybrid template defines:

1. `primary_research` — potential original experimental research.
2. `animal_study` — potential living non-human animal population.
3. `in_vivo_experiment` — potential in vivo experimental intervention.
4. `depression_model` — potential depression model or depressive-like phenotype.
5. `control_group` — an exploratory comparator assessment.
6. `eligible_outcomes` — an exploratory outcome assessment.
7. `overall_eligibility` — a separately asked, integrated retention decision.

Both designs favour retention when information is missing or ambiguous. In the supplied hybrid draft, control-group and outcome judgments must not independently cause exclusion in the primary classification. The overall question is not currently calculated by AND-ing, summing or multiplying the component probabilities. An aggregation rule would require protocol review; the probabilities do not establish independence or a joint eligibility probability.

The source designs are [binary_noul.json](../prompts/drafts/binary_noul.json) and [hybrid_noul_corrected.json](../prompts/drafts/hybrid_noul_corrected.json). The original truncated hybrid draft is retained for provenance and is not executable. The corrected draft's `jev-latest` placeholder is not either benchmark's configured model identifier.

**Scope for review:** either model could be assessed under a separately implemented hybrid workflow. That would introduce a second experimental factor—model × question design. The current runner's one-question safeguards have not been changed. Hybrid execution, aggregation and primary scoring must first be specified; criterion-level accuracy would additionally need criterion-level human reference labels. Overall human retention labels alone cannot validate all six component judgments.

### 4.3 The eligibility question actually used in these runs

These are the exact screening instructions stored with the model runs. Both models used the same task fingerprint. The synthetic connection-test question from notebook 02 is not substituted for the evaluation criteria.

The positive class means **retain for full-text assessment**, including uncertain eligibility; it is not a diagnosis or a final inclusion decision. The workflow maps human `Inclusion Status` 1 to retain and 0 to exclude. Number of human screeners, independence of screening and adjudication procedures still need author confirmation; they cannot be inferred from the CSV.

In [ ]:
print("Archived task SHA-256:", summary.provenance["task_sha256"])
for question in summary.tables["archived_criteria"].itertuples(index=False):
    print("\nQuestion:", question.question_id)
    print(question.instructions)
    print("Retain:", question.retain)
    print("Exclude:", question.exclude)

## 5. Hosted model execution and recorded exceptions

Requested app identifiers and provider-returned model names are shown separately. UTC start/end times, recorded attempts and reported input tokens come from the original run records. `completed_with_exceptions` is preserved as recorded; missing-text cases are not silently reclassified as completed predictions.

An app identifier or model name does not establish an immutable model-weight revision. Provider-reported revision metadata is shown where present, while verification remains false. Token counts are not monetary costs; provider billing/pricing is not established by this report.

In [ ]:
execution_columns = [
    "run_id",
    "requested_app",
    "returned_models",
    "status",
    "started_at",
    "ended_at",
    "recorded_attempts",
    "reported_input_tokens",
    "provider_reported_revisions",
    "immutable_revision_verified",
    "unreported_revision",
]
display(summary.tables["model_execution"][execution_columns].T)

## 6. Exploratory thresholds and screening performance

The table compares the highest observed cutoffs meeting the configured recall target, the illustrative rounded cutoffs in the summary YAML and a 0.5 baseline. All use **retain if P(true) >= cutoff**, including ties. The rounded cutoffs are discussion options, not locked validation settings. The official analysis threshold remains whatever is explicitly recorded in `analysis_full.yaml`.

`recall_scored` uses scored human-retain studies. `recall_cohort_lower_bound` divides scored true positives by **all** human-retain studies and gives unscored cases no model credit. Specificity, PPV (= precision) and F2 use scored labelled pairs. `review_queue_with_unscored` includes all unavailable predictions.

F2 is `5 TP / (5 TP + 4 FN + FP)`. WSS@95 is the excluded fraction minus 0.05, shown only where observed recall reaches 95% in the stated scope. Cohort-conservative WSS additionally requires complete reference labels and includes all unscored records in review workload. These calculations follow notebook 04; undefined/unattainable results remain unavailable.

**Interpretation:** cutoff selection and performance measurement use the same human labels. These are exploratory estimates without confidence intervals or independent validation. “95%” here is a recall target, not a confidence level or guarantee for new studies. See the [WSS definition](https://academic.oup.com/jamia/article-pdf/13/2/206/2178574/13-2-206.pdf) and [validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html).

In [ ]:
print("Exploratory recall target:", summary.settings.target_recall)
print("Illustrative cutoffs:", summary.settings.illustrative_cutoffs)
print("Official configured analysis threshold:", summary.data.settings.decision_threshold)
metric_columns = [
    "model",
    "cutoff_source",
    "threshold",
    "recall_scored",
    "recall_cohort_lower_bound",
    "specificity_scored",
    "ppv_scored",
    "f2_scored",
    "wss95_scored",
    "wss95_cohort_conservative",
    "review_queue_with_unscored",
]
performance = summary.tables["screening_metrics"].reindex(columns=metric_columns)
display(
    performance.style.format(
        {
            "threshold": "{:.17g}",
            "recall_scored": "{:.2%}",
            "recall_cohort_lower_bound": "{:.2%}",
            "specificity_scored": "{:.2%}",
            "ppv_scored": "{:.2%}",
            "f2_scored": "{:.3f}",
            "wss95_scored": "{:.2%}",
            "wss95_cohort_conservative": "{:.2%}",
            "review_queue_with_unscored": "{:.0f}",
        },
        na_rep="not available",
    )
)

## 7. Figures and paired false-negative audit

The curves show the recall/workload tradeoff across saved scores. Solid recall curves use the full-labelled lower bound; dotted curves use scored pairs. Vertical lines and matrices use the exact target-based candidate cutoffs, which can differ from the rounded examples.

The matrices contain scored labelled pairs only; unscored records remain outside them. The paired-error table instead uses the **illustrative rounded cutoffs** and counts human-retain studies missed by both models, only one model or neither. Unavailable scores are reported separately. These overlapping-error counts help direct private error review; they do not establish a statistically significant model difference.

In [ ]:
from jev_eval.evaluation import threshold_tradeoff_curve
from jev_eval.reporting import plot_recall_candidate_matrices, plot_threshold_tradeoffs

figures = {}
figures["recall-workload"] = plot_threshold_tradeoffs(
    threshold_tradeoff_curve(summary.data), summary.candidates, summary.settings.target_recall
)
display(figures["recall-workload"])
candidate_matrix = plot_recall_candidate_matrices(summary.candidates)
if candidate_matrix is not None:
    figures["candidate-confusion-matrices"] = candidate_matrix
    display(candidate_matrix)
else:
    print(
        "The requested recall target is unattainable with available scores; no candidate matrix is shown."
    )
display(summary.tables["paired_positive_errors"].T.rename(columns={0: "value"}))

## 8. Reproducibility record and methods draft

Original inference fingerprints, code/configuration identities and environment metadata remain archived separately from this report's analysis environment. A changed analysis implementation does not imply that new model requests were made. Missing historical Git commits or immutable weight revisions remain unknown.

The methods paragraph is a draft assembled from the saved evidence. Authors should confirm human screening and adjudication methods and review the audit findings before using it in a paper. The optional export includes complete report provenance and output checksums.

In [ ]:
provenance_keys = [
    "report_type",
    "generated_at",
    "workbook_sha256",
    "conversion_manifest_sha256",
    "states_sha256",
    "reference_sha256",
    "task_sha256",
    "summary_config_sha256",
    "analysis_config_sha256",
    "validation_results_present",
    "immutable_weight_revision_verified",
]
display(pd.DataFrame([{"item": key, "value": summary.provenance[key]} for key in provenance_keys]))
display(summary.tables["model_execution"][["run_id", "git_commit", "python", "lockfile_sha256"]])
display(Markdown("**Methods draft — author review required**\n\n" + summary.methods))

## 9. Colleague review and the planned unseen-study validation

This report marks colleague review as pending and validation as planned. It does not certify either as completed.

Before validation, agree and document:

1. The research question, archived eligibility criteria and human-reference/adjudication procedures, including the workbook's label disagreements.
2. The primary model comparison, model-specific cutoffs, inclusive cutoff rule, recall target and treatment of unavailable predictions.
3. The primary metrics, denominators, confidence-interval methods and rules for reporting failure to meet the recall target.
4. A versioned protocol and analysis snapshot locked before inspecting validation outcomes. Cutoffs chosen on the present cohort must not be retuned on validation labels while retaining a confirmatory claim.
5. Checks that validation studies do not overlap the development cohort, including duplicate/linked records and overlapping titles where appropriate. “Unseen” means unused for this analysis and threshold selection; provider training-data exposure is not established.
6. Separate immutable validation artifacts and run identifiers. The new cohort must not overwrite these development runs.

The planned validation will be a separate workflow after colleague review. Until then, conclusions should describe retrospective exploratory performance. Confidence intervals, calibration analyses and confirmatory tests have not been added to this interim snapshot.

In [ ]:
display(
    Markdown(
        f"**Next stage:** review this interim report with a colleague, then validate the locked "
        f"protocol on **{summary.settings.planned_validation_records:,} unseen studies**. "
        "No validation cohort has been loaded or evaluated by this notebook."
    )
)

## 10. Optional aggregate report export — writes files

**File-writing warning:** the next cell writes a new private report directory only if you set both `allow_writes: true` in the summary YAML and `EXPORT_INTERIM_REPORT = True` below. It makes no API calls and does not modify archived inference or data files.

The bundle contains a self-contained HTML summary with embedded figures, aggregate CSV tables, PNG/PDF figures, a methods draft and a provenance/checksum manifest. It excludes citation text, row-level identifiers, raw responses and credentials. Generated reports stay under ignored `results/summary/interim/`; review the report before sharing it. Save any executed notebook copy under ignored `results/notebooks/` and clear outputs from this source notebook before Git.

In [ ]:
from jev_eval.summary import export_interim_summary

EXPORT_INTERIM_REPORT = False
if EXPORT_INTERIM_REPORT:
    report_directory = export_interim_summary(summary, figures, allow_writes=True)
    print("Private report bundle:", report_directory)
    print("Open summary.html for the self-contained colleague-review report.")
else:
    print("Export skipped. All tables and figures remain in memory; no report files written.")